# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umarfarukh786/FlyRank-task1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The queue is a review order, not an automatic publishing order. Higher `final_refresh_score` means the item has stronger observed evidence for manual review in this starter slice. Reason codes make the score inspectable: they identify the specific pattern that contributed to the suggested action.

The action ladder is:

| Archetype or reason | Suggested action | Human question |
|---|---|---|
| `thin_visible_page` | `expand_and_refresh` | Is the page too shallow for the demand it already receives, and what useful coverage is missing? |
| `low_ctr_visible_page` + decline risk | `refresh_and_review_ctr` | Is the result visible but mismatched to search intent or poorly represented in the snippet? |
| `low_engagement_visible_page` + decline risk | `refresh_and_review_engagement` | Does the page satisfy the visitor after the click, or is the measurement context unusual? |
| `stale_visible_page` or `declining_with_demand` | `refresh` | Is there a current, evidence-based update that a subject-matter reviewer can justify? |
| no strong reason code | `monitor` | Is there enough evidence to wait for another observation window? |

The first 20 rows are the practical review queue. A reason code is a prompt for investigation, not proof that a page needs a particular edit.

In [7]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT / "scripts"))

FEATURE_SOURCE = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
BASELINE_SOURCE = ROOT / "data" / "processed" / "baseline_refresh_queue.csv"
QUEUE_SOURCE = ROOT / "outputs" / "refresh_queue.csv"

if not FEATURE_SOURCE.exists() or not BASELINE_SOURCE.exists():
    print("Prepared inputs not found; building the feature vector and baseline first.")
    for script_name in ("01_prepare_features.py", "02_baseline_score.py"):
        subprocess.run(
            [sys.executable, str(ROOT / "scripts" / script_name)],
            cwd=ROOT,
            check=True,
        )

if not QUEUE_SOURCE.exists():
    print("Validated queue not found; training and exporting the queue first.")
    for script_name in ("03_train_model.py", "04_evaluate_and_export.py"):
        subprocess.run(
            [sys.executable, str(ROOT / "scripts" / script_name)],
            cwd=ROOT,
            check=True,
        )

if not QUEUE_SOURCE.exists():
    raise FileNotFoundError(
        "Unable to create outputs/refresh_queue.csv from the reference pipeline."
    )

queue = pd.read_csv(QUEUE_SOURCE)
required_columns = {
    "final_rank", "content_id", "final_refresh_score", "confidence",
    "suggested_action", "final_reason_codes",
}
missing_columns = required_columns.difference(queue.columns)
if missing_columns:
    raise ValueError(f"Queue is missing required columns: {sorted(missing_columns)}")

reason_labels = {
    "model_decline_risk": "model decline risk",
    "visible_model_opportunity": "visible demand",
    "declining_with_demand": "observed decline with demand",
    "stale_visible_page": "stale page with visibility",
    "thin_visible_page": "thin page with visibility",
    "low_ctr_visible_page": "low CTR with visibility",
    "low_engagement_visible_page": "low engagement",
    "ctr_review_candidate": "CTR review candidate",
    "engagement_review_candidate": "engagement review candidate",
    "general_refresh_review": "general review",
}

def first_matching_reason(reason_text):
    reasons = str(reason_text).split("|")
    for reason in [
        "thin_visible_page",
        "low_ctr_visible_page",
        "low_engagement_visible_page",
        "stale_visible_page",
        "declining_with_demand",
    ]:
        if reason in reasons:
            return reason
    return "general_refresh_review"

queue["primary_archetype"] = queue["final_reason_codes"].map(first_matching_reason)
queue["reason_summary"] = queue["final_reason_codes"].map(
    lambda text: "; ".join(reason_labels.get(reason, reason) for reason in str(text).split("|"))
)

print(f"Rows in validated queue: {len(queue):,}")
print("Suggested action counts:")
display(queue["suggested_action"].value_counts().rename_axis("action").to_frame("rows"))
print("Top 20 review queue:")
display(
    queue.head(20)[
        ["final_rank", "final_refresh_score", "confidence", "suggested_action", "reason_summary"]
    ]
)

Rows in validated queue: 30,000
Suggested action counts:


,rows
action,
monitor,13069
refresh,8207
refresh_and_review_ctr,6655
refresh_and_review_engagement,1987
expand_and_refresh,82


Top 20 review queue:


,final_rank,final_refresh_score,confidence,suggested_action,reason_summary
0,1,81.928467,high,refresh_and_review_ctr,observed decline with demand; low CTR with vis...
1,2,81.728449,high,refresh_and_review_ctr,observed decline with demand; low CTR with vis...
2,3,81.639118,medium,refresh_and_review_ctr,observed decline with demand; low CTR with vis...
3,4,80.804986,medium,refresh_and_review_ctr,observed decline with demand; low CTR with vis...
4,5,80.801530,high,refresh_and_review_ctr,observed decline with demand; low CTR with vis...
5,6,80.752578,high,refresh_and_review_ctr,observed decline with demand; low CTR with vis...
6,7,80.632372,high,refresh_and_review_ctr,observed decline with demand; low CTR with vis...
7,8,80.439403,medium,refresh,observed decline with demand; model decline ri...
8,9,80.204325,high,refresh_and_review_ctr,observed decline with demand; low CTR with vis...
9,10,80.146808,high,refresh_and_review_ctr,observed decline with demand; low CTR with vis...


## 2. Intended use and limits

**Intended use:** an editor or SEO/content reviewer uses the ranked queue to decide which anonymized content items to inspect first. The score compresses observed traffic, freshness, position, engagement, and model signals into a review order. The reason codes tell the reviewer what to check before choosing an action.

**Limits:** this is decision-support on a bundled anonymized starter snapshot. The decline label is a proxy derived from the current trend window, not a guaranteed future outcome. The model does not estimate the causal effect of refreshing a page, does not know editorial quality or business context, and does not contain titles, URLs, client names, or page content. A high score means “review earlier,” not “change this page automatically.”

**Cost/value framing:** review the highest-confidence, high-visibility rows first when reviewer time is scarce, because the cost of inspecting them is more likely to be justified by measurable demand. Use `monitor` for lower-evidence rows and avoid expensive rewrites until a human confirms the issue. Impressions, sessions, and position are prioritization context, not a promise of financial value; actual value requires a later intervention and outcome measurement.

In [9]:
use_summary = pd.DataFrame([
    {
        "question": "Who uses it?",
        "answer": "A human content or SEO reviewer deciding which anonymized items to inspect first.",
    },
    {
        "question": "What does a high score mean?",
        "answer": "Higher observed priority for review in this snapshot, with reason codes to investigate.",
    },
    {
        "question": "What does it not mean?",
        "answer": "It is not a causal refresh recommendation, a traffic guarantee, or an automatic publish/delete decision.",
    },
    {
        "question": "What should be checked?",
        "answer": "Current page intent, accuracy, search context, business importance, recent changes, and measurement quality.",
    },
])
display(use_summary)
print("Confidence distribution:")
display(queue["confidence"].value_counts(normalize=True).mul(100).round(1).rename("percent").to_frame())

,question,answer
0,Who uses it?,A human content or SEO reviewer deciding which...
1,What does a high score mean?,Higher observed priority for review in this sn...
2,What does it not mean?,"It is not a causal refresh recommendation, a t..."
3,What should be checked?,"Current page intent, accuracy, search context,..."


Confidence distribution:


,percent
confidence,
low,50.0
medium,38.1
high,11.9


## 3. Human review + the no-go list

Before acting on any row, a reviewer should:

1. Confirm the page identity and current business importance in an approved internal system.
2. Check whether the reason code matches the current page, intent, and search context.
3. Verify that the signal is not caused by tracking gaps, a migration, seasonality, a recent launch, or a known measurement change.
4. Choose the smallest defensible action and record the evidence and reviewer decision.
5. Recheck the recommendation after the next observation window rather than assuming the action worked.

**No-go list:** do not auto-publish edits, delete or redirect pages, change claims in regulated or safety-sensitive content, contact clients, infer search-engine causality, expose pseudonymous IDs outside approved workflows, or treat a model score as approval to spend editorial budget. Those decisions require a qualified human and context that this dataset does not contain.

In [12]:
review_rules = pd.DataFrame([
    {
        "rule_type": "review required",
        "rule": "Every high-confidence row is inspected by a person before any change.",
    },
    {
        "rule_type": "review required",
        "rule": "The primary reason code must be confirmed against current page and measurement context.",
    },
    {
        "rule_type": "no-go",
        "rule": "Never auto-publish, delete, redirect, contact a client, or claim causal traffic impact from the score.",
    },
    {
        "rule_type": "no-go",
        "rule": "Do not use the queue to make decisions about regulated, safety-sensitive, or legally consequential content without specialist review.",
    },
])
display(review_rules)
print(f"Rows requiring highest-priority review: {(queue['confidence'] == 'high').sum():,}")
print("No-go rules are policy constraints, not model outputs.")

,rule_type,rule
0,review required,Every high-confidence row is inspected by a pe...
1,review required,The primary reason code must be confirmed agai...
2,no-go,"Never auto-publish, delete, redirect, contact ..."
3,no-go,Do not use the queue to make decisions about r...


Rows requiring highest-priority review: 3,576
No-go rules are policy constraints, not model outputs.


## 4. Monitoring / retrain triggers

The queue should be treated as a snapshot. Monitor it after each refresh cycle and retrain or revisit the feature contract when the population, label, or decision use changes. Useful triggers are: the observed decline rate moves by more than 5 percentage points, Precision@50 falls below the recorded baseline on a labeled follow-up sample, a client or content-type group loses coverage, missingness changes by more than 10 percentage points, or the action mix shifts by more than 20% without an editorial explanation. A new label window or a new warehouse release requires a fresh temporal leakage review before reuse.

In [14]:
action_share = queue["suggested_action"].value_counts(normalize=True)
confidence_share = queue["confidence"].value_counts(normalize=True)
monitoring_triggers = pd.DataFrame([
    {
        "trigger": "Observed label rate changes by >5 percentage points",
        "current_snapshot": f"{queue['is_declining_label'].mean():.3f}" if "is_declining_label" in queue.columns else "not available in paper-safe queue",
        "response": "Recheck label definition, time window, and retrain before comparing scores.",
    },
    {
        "trigger": "Precision@50 falls below the recorded baseline on a later labeled sample",
        "current_snapshot": "requires follow-up labels",
        "response": "Pause score-led prioritization; inspect errors and retrain or revise features.",
    },
    {
        "trigger": "Any action share moves by >20% from the prior run",
        "current_snapshot": "; ".join(f"{key}={value:.2f}" for key, value in action_share.items()),
        "response": "Check population changes, missingness, seasonality, and pipeline version.",
    },
    {
        "trigger": "Feature missingness changes by >10 percentage points",
        "current_snapshot": "requires feature-profile comparison",
        "response": "Investigate instrumentation or source changes before using the queue.",
    },
    {
        "trigger": "New clients, label window, or warehouse release",
        "current_snapshot": f"{queue['content_id'].nunique():,} unique content items in this run",
        "response": "Repeat grouped/time-aware validation and leakage review.",
    },
])
display(monitoring_triggers)
print("Monitoring is a review trigger, not an automatic production retraining system.")

,trigger,current_snapshot,response
0,Observed label rate changes by >5 percentage p...,0.542,"Recheck label definition, time window, and ret..."
1,Precision@50 falls below the recorded baseline...,requires follow-up labels,Pause score-led prioritization; inspect errors...
2,Any action share moves by >20% from the prior run,monitor=0.44; refresh=0.27; refresh_and_review...,"Check population changes, missingness, seasona..."
3,Feature missingness changes by >10 percentage ...,requires feature-profile comparison,Investigate instrumentation or source changes ...
4,"New clients, label window, or warehouse release","30,000 unique content items in this run",Repeat grouped/time-aware validation and leaka...


Monitoring is a review trigger, not an automatic production retraining system.


## 5. Exports for the paper

The notebook writes a paper-safe ranked queue and metrics receipt under `work/outputs/`. It also writes reusable SVG figures under `work/figures/`. The queue is regenerated rather than committed; the metrics JSON and figures are the traceable artifacts for the paper. The exported queue omits client IDs and the label column, while retaining the anonymized content ID needed to join an approved internal review workflow.

In [17]:
import json

import matplotlib.pyplot as plt

OUTPUT_DIR = ROOT / "work" / "outputs"
FIGURE_DIR = ROOT / "work" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

paper_columns = [
    "final_rank", "content_id", "final_refresh_score", "best_model_name",
    "best_model_probability", "confidence", "suggested_action", "final_reason_codes",
    "impressions_90d", "sessions_90d", "avg_position", "ctr", "content_age_days",
    "days_since_last_update", "word_count", "trend_direction",
]
paper_queue = queue[[column for column in paper_columns if column in queue.columns]].copy()
queue_path = OUTPUT_DIR / "action_playbook_queue.csv"
paper_queue.to_csv(queue_path, index=False)

reason_counts = {}
for reason_text in queue["final_reason_codes"].astype(str):
    for reason in reason_text.split("|"):
        reason_counts[reason] = reason_counts.get(reason, 0) + 1

metrics = {
    "source": "outputs/refresh_queue.csv",
    "rows_scored": int(len(queue)),
    "top_20_actions": queue.head(20)["suggested_action"].value_counts().to_dict(),
    "action_counts": {str(key): int(value) for key, value in queue["suggested_action"].value_counts().items()},
    "confidence_counts": {str(key): int(value) for key, value in queue["confidence"].value_counts().items()},
    "reason_code_counts": {str(key): int(value) for key, value in reason_counts.items()},
    "top_score": float(queue["final_refresh_score"].max()),
    "high_confidence_rows": int((queue["confidence"] == "high").sum()),
    "intended_use": "human review prioritization and decision-support",
    "not_a_claim": "not causal, not automatic publishing, not a future traffic guarantee",
}
metrics_path = OUTPUT_DIR / "action_playbook_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True), encoding="utf-8")

def save_bar(values, title, filename, color):
    ordered = values.sort_values(ascending=False)
    figure, axis = plt.subplots(figsize=(8, 4.5))
    ordered.plot(kind="bar", ax=axis, color=color)
    axis.set_title(title)
    axis.set_ylabel("Rows")
    axis.set_xlabel("")
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / filename, format="svg")
    plt.close(figure)

save_bar(queue["suggested_action"].value_counts(), "Action mix", "action_mix.svg", "#426B69")
save_bar(queue["confidence"].value_counts().reindex(["high", "medium", "low"]).dropna(), "Confidence mix", "confidence_mix.svg", "#4E79A7")
save_bar(pd.Series(reason_counts), "Reason code mix", "reason_code_mix.svg", "#B07AA1")

print(f"Wrote queue: {queue_path}")
print(f"Wrote metrics: {metrics_path}")
print(f"Wrote figures: {FIGURE_DIR}")

Wrote queue: e:\Comm_Project\internship\FlyRank-task1\work\outputs\action_playbook_queue.csv
Wrote metrics: e:\Comm_Project\internship\FlyRank-task1\work\outputs\action_playbook_metrics.json
Wrote figures: e:\Comm_Project\internship\FlyRank-task1\work\figures


## Self-check

Before submitting, rerun the notebook manually and confirm:

- [ ] The queue is ranked by the validated final score and every row has a reason code and suggested action.
- [ ] The archetype-to-action mapping is visible and each action requires human review.
- [ ] Intended use, limits, cost/value considerations, and no-go cases are explicit.
- [ ] Monitoring and retrain triggers are concrete enough to act on.
- [ ] The queue and metrics JSON are written under `work/outputs/`.
- [ ] Reusable figures are written under `work/figures/`.
- [ ] No client names, URLs, private queries, credentials, or automatic publishing claims appear.
- [ ] Claims use observed, measured, directional, or decision-support language.
- [ ] The notebook runs top to bottom with no errors.
- [ ] The notebook is committed under `work/notebooks/w07_action_playbook.ipynb`.